<img src="logo.png" alt="Vegeta" width="240">

# Project Onager — Onager Sweeper, the street cleaner

The **Sweeper** is the Onager series' outdoor cleaning unit, after the pattern of the compact indoor scrubbers
(a white rounded body, a display on top) but built for the street: the Sentinel's wheel-leg chassis (notebook 20)
set **low in a fixed stance** — no active suspension, the leg modules' brakes hold the ride height — under a
**bulky rounded body**; a **rotary disc broom** under the hull centre with a **suction hood** right behind it (a
square duct up to the fan and the hopper); the Manus's two **pincer arms** (notebook 22) on the front of the roof
for what the broom cannot take — a bottle, a brick, a branch — and a **basket** on the back of the roof they drop
it into.

The job it is sized for (inputs): a street with litter (butts, leaves, paper, caps, packets, cans, gravel) swept at
1 m/s, with the odd obstacle — a brick — and bulky rubbish — a box — in the way. The notebook checks the design
end to end:

```
CAD (Dedalus): the body, the broom, the hood and duct, the basket, the arms ─► masses, CG, the arms' reach
the sweeping gear: broom speed and torque, the fan's flow and pressure (actuators.py), power and endurance
CFD (Aeromant, OpenFOAM — the new suction_hood template): the air under the hood — the depression the fan must
    supply, the inflow at the lips, the upward speed in the hood; which litter it lifts; litter particles in the field
FEA (Talos): the basket loaded, the hood under the depression; their modes against the broom's and fan's unbalance
MuJoCo (ChironLab): the street — litter the hood vacuums (the CFD's drag on Chiron props), a brick and a box the
    pincers load into the basket — with a movie
```

In [ ]:
import json, math, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vegeta import dedalus, talos, chiron, aeromant
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.chiron import viz as cviz
from vegeta.aeromant import viz as aviz
sys.path.insert(0, "designs")
import actuators as act
import onager_robot as orb, onager_controller as oc, onager_manus_robot as omr
import onager_sweeper_robot as osr, onager_sweeper_cfd as cfd, onager_sweeper_scenario as oss, onager_sweeper_controller as osc

RUNS = Path("_runs/onager_sweeper"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
OUT = Path("output/23_onager_sweeper"); OUT.mkdir(parents=True, exist_ok=True)
for f in ("onager.py", "onager_manus.py", "onager_sweeper.py"):
    shutil.copy(Path("designs") / f, RUNS / f)
sys.path.insert(0, str(RUNS))
design = dedalus.load_design(f"{RUNS / 'onager_sweeper.py'}:OnagerSweeper")
G = 9.81
JOB = pd.Series({"sweeping speed [m/s]": 1.0, "hood width [m]": osr.SWEEPER["hood_width"] / 1000, "fan flow [m³/s]": osr.FAN_FLOW,
                 "broom speed [rpm]": osr.BROOM_RPM, "litter classes": ", ".join(cfd.DEBRIS),
                 "obstacle": f"brick {oss.OBJECTS['brick']['mass']} kg", "bulky rubbish": f"box {oss.OBJECTS['box']['mass']} kg"},
                name="Onager Sweeper — the job (inputs)")
display(JOB.to_frame())
pd.DataFrame(design.params.table()).set_index("name").loc[list(osr.SWEEPER)]

## 1. The machine

The Sweeper changes the Sentinel's chassis (`onager_sweeper.SWEEPER_CHASSIS`: the hull 2.2 × 1.0 × 0.65 m with
its floor 300 mm over the road, filleted edges, shorter legs in a 40°/50° stance, no turret, the display post on
the nose) and the Manus's arms (further back, longer links: a pincer must reach the road ahead of the front wheels
*and* the basket 1.1 m behind the shoulders). Its own parts: the disc broom (Ø 600, PA6 disc, PP bristles) ahead
of the hull centre, the hood (500 × 150 mm, lips 30 mm over the road, the front lip a rubber flap) right behind it
with its 140 mm square duct, the basket (700 × 900 × 400 mm, 2 mm Al 5754) on the back of the roof.

In [ ]:
p = design.resolve()
parts = {k: design.generate(part=k) for k in ("robot", "hull", "hood", "hood_shell", "basket", "broom_disc")}
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.2) for k, g in parts.items()}
sg = osr.sweeper_geometry()
ag = omr.arm_geometry(osr.design_params())
print(f"hull floor {p['hull_bottom'] / 1000:.2f} m over the road, roof at {(p['hull_bottom'] + p['hull_height']) / 1000:.2f} m | shoulders {ag['z'] + sg['hull_z']:.2f} m up, "
      f"{ag['x']:.2f} m ahead of the hull centre | arm reach {ag['Lu'] + ag['Lf'] + ag['Lp'] + ag['Lj']:.2f} m | basket rim {sg['basket_world'][5]:.2f} m up")
dviz.show(dviz.plot3d(parts["robot"]))

In [ ]:
dviz.show(dviz.plot3d(parts["hood"]))

In [ ]:
cad_num = osr.cad_numbers(osr.design_params(), recompute=True)
assert max(abs(cad_num[k] / osr.CAD[k] - 1) for k in osr.CAD) < 0.01
budget = osr.mass_budget(cad=cad_num)
mass = pd.DataFrame({"mass [kg]": budget}); M = mass["mass [kg]"].sum(); mass.loc["TOTAL"] = M
gear = pd.DataFrame({"sweeping gear [kg]": osr.sweeper_masses(cad=cad_num)}); gear.loc["TOTAL"] = gear.iloc[:, 0].sum()
robot = osr.sweeper(cad=cad_num)
lab_flat = osr.sweeper_lab(chiron.Flat(), robot=robot)
ep_stand = lab_flat.run(oc.Stand(), duration=2.0, rules=None, settle=0.0)
fz = np.asarray(ep_stand.log["foot_force"])[-1, :, 2]
com = np.asarray(ep_stand.log["com"])[-1]
print(f"Sweeper {M:.0f} kg (Sentinel 408, Manus 458) | CG {com[0] * 1000:+.0f} mm from the hull centre, {com[2]:.2f} m up | battery at x = {robot.battery_x:+.2f} m | wheels {fz.round(0)} N")
display(gear.round(2))
mass.round(1)

## 2. The sweeping gear: broom, fan, power and endurance

**Broom.** A disc broom works by its bristle tips: pressed on the road with a set force (an input, 100 N — the
mount's spring), they drag with μ ≈ 0.5 at the mean bristle radius, so the torque is `μ N r` and the power
`μ N r ω`. The catalogue's `broom drive 1.5 kW` runs it at 150 rpm.

**Fan.** The CFD of §4 gives the depression at the duct; the hopper, the filter and the ducting downstream add
their losses (an input: 1.5 kPa, a cyclone-and-filter class). The fan's air power is `Q Δp`; with a fan
efficiency of 0.55 and the motor's 0.9 (inputs) the `suction fan 4 kW` has margin to spare.

**Endurance** on the Sentinel's 6 kWh pack at the sweeping speed: rolling resistance, broom, fan, electronics.

In [ ]:
BROOM = act.get(osr.BROOM_DRIVE); FAN = act.get(osr.FAN_MOTOR)
N_bristle, mu_bristle, r_eff = 100.0, 0.5, 0.8 * sg["broom_r"]            # inputs: bristle load, friction, mean bristle radius
w_broom = osr.BROOM_RPM * 2 * math.pi / 60
tau_broom = mu_bristle * N_bristle * r_eff; P_broom = tau_broom * w_broom
tau_avail = BROOM.stall_Nm * (1 - osr.BROOM_RPM / BROOM.no_load_rpm)
dp_downstream, eta_fan, eta_motor = 1500.0, 0.55, 0.9                     # inputs
dp_fan = cfd.SUCTION["fan_static_pressure_Pa"] + dp_downstream
P_air = osr.FAN_FLOW * dp_fan; P_fan_shaft = P_air / eta_fan; P_fan_el = P_fan_shaft / eta_motor
w_fan = 4500 * 2 * math.pi / 60; tau_fan = P_fan_shaft / w_fan
P_roll = orb.C_RR * M * G * 1.0; P_elec = 150.0
P_total = P_roll + P_broom / 0.85 + P_fan_el + P_elec
E_usable = 6.0e3 * 0.9
hours = E_usable / P_total
power = pd.DataFrame({"W": {"rolling resistance at 1 m/s (C_RR 0.03)": P_roll, "broom (bristles 100 N, μ 0.5, 150 rpm; drive η 0.85)": P_broom / 0.85,
                            f"fan ({osr.FAN_FLOW} m³/s at {dp_fan:.0f} Pa; η 0.55 × 0.9)": P_fan_el, "computer, sensors, display": P_elec, "TOTAL": P_total}})
print(f"broom: {tau_broom:.1f} N·m, {P_broom:.0f} W at {osr.BROOM_RPM:.0f} rpm — the drive has {tau_avail:.0f} N·m there (rated {BROOM.rated_Nm:.0f} N·m continuous)")
print(f"fan: Δp {dp_fan:.0f} Pa (duct {cfd.SUCTION['fan_static_pressure_Pa']:.0f} + downstream {dp_downstream:.0f}), air {P_air:.0f} W, shaft {P_fan_shaft:.0f} W = {tau_fan:.1f} N·m at 4500 rpm (motor rated {FAN.rated_Nm:.0f} N·m)")
print(f"endurance: {E_usable / 1000:.1f} kWh usable / {P_total / 1000:.2f} kW = {hours:.1f} h — {hours * 3600 * 1.0 / 1000:.1f} km of street, {hours * 3600 * 1.0 * sg['hood_world'][4] * 2 / 10000:.1f} ha of a {2 * sg['hood_world'][4]:.1f} m lane")
power.round(0)

## 3. The arms: reach and torques

The Manus's inverse kinematics (`onager_manus_robot.arm_ik`) with the Sweeper's longer links: the pin must reach
the brick on the road 0.75 m ahead of the shoulders (ahead of the front wheels), pincer down, and the drop point
over the basket — 1.1 m behind the shoulders, 0.33 m above the rim: the pedestals turn through 180° (a slip ring).
The static joint torques holding the brick at the pick and at the drop against the modules.

In [ ]:
down = -math.pi / 2
bz1 = sg["basket"][5]
targets = {"brick on the road (pick)": ((ag["x"] + 0.75, 0.0, 0.07 + osc.PICK["brick"]["grip_depth"] - sg["hull_z"]), "L"),
           "box on the road (pick)": ((ag["x"] + 0.75, 0.0, 0.15 + osc.PICK["box"]["grip_depth"] - sg["hull_z"]), "R"),
           "drop into the basket (L)": ((-0.45, ag["y"] + 0.02, bz1 + 0.36), "L"), "drop into the basket (R)": ((-0.45, -ag["y"] - 0.02, bz1 + 0.36), "R")}
m_arm = omr.arm_masses(osr.design_params(), cad_num)
stall = {j: act.get(omr.ARM_ACTUATORS[j]).stall_Nm for j in ("shoulder", "elbow", "wrist")}
def torques(q, load_kg):
    _, sh, el, wr = q
    p1, p2, p3 = -sh, -(sh + el), -(sh + el + wr)
    elbow = ag["Lu"] * math.cos(p1); wrist = elbow + ag["Lf"] * math.cos(p2); pin = wrist + ag["Lp"] * math.cos(p3); jaw_c = pin + 0.1 * math.cos(p3)
    items = [(m_arm["upper arm (Al 6082, CAD)"], ag["Lu"] / 2 * math.cos(p1)), (m_arm["elbow drive (harmonic 150 Nm, brake)"], elbow),
             (m_arm["forearm (Al 6082, CAD)"], elbow + ag["Lf"] / 2 * math.cos(p2)), (m_arm["wrist drive (harmonic 60 Nm, brake)"], wrist),
             (m_arm["palm (wrist housing, screw mount)"] + m_arm["jaw drive (jaw screw 6 kN)"], wrist + ag["Lp"] / 2 * math.cos(p3)),
             (m_arm["jaws 2x (tool steel, CAD)"], jaw_c), (load_kg, jaw_c)]
    return {"shoulder": abs(G * sum(mi * xi for mi, xi in items)), "elbow": abs(G * sum(mi * (xi - elbow) for mi, xi in items[2:])),
            "wrist": abs(G * sum(mi * (xi - wrist) for mi, xi in items[4:]))}
rows = {}
for name, (tgt, side) in targets.items():
    q = omr.arm_ik(tgt, down, side, osr.design_params())
    load = oss.OBJECTS["brick"]["mass"] if "brick" in name or "(L)" in name else oss.OBJECTS["box"]["mass"]
    row = {"reachable": q is not None, "yaw [deg]": math.degrees(q[0]) if q else float("nan"), "extension": math.hypot(*(np.array(tgt[:2]) - np.array([ag["x"], omr.SIDES[side] * ag["y"]]))) and float("nan")}
    if q:
        tq = torques(q, load)
        d = math.sqrt((tgt[0] - ag["x"]) ** 2 + (tgt[1] - omr.SIDES[side] * ag["y"]) ** 2 + (tgt[2] - ag["z"]) ** 2)
        row["extension"] = d / (ag["Lu"] + ag["Lf"])
        row.update({f"{j} / stall": tq[j] / stall[j] for j in stall})
    rows[name] = row
pd.DataFrame(rows).T.round(2)

## 4. The suction in CFD (Aeromant, OpenFOAM): the air under the hood

Aeromant's new `suction_hood` template: the air between the road (a wall moving backwards at the sweeping speed)
and the hull floor, the hood with its lips a gap over the road, the broom disc and housing ahead of it, the duct
rising through the floor where a patch draws the fan's flow; the sides are the atmosphere. Steady k-ω SST, here on
the **fast** preset (`onager_sweeper_cfd.QUALITY`: ~110 k cells, the gap under the lips two cells deep, 150
iterations — the depression and the mean speeds within ~20 %; `quality="fine"` is 0.9 M cells and 5 mm in the
gap). It gives the **depression at the duct** (what the fan must
supply, before the hopper and filter), the **inflow at the lips** and the **upward speed in the hood**, which decide
what the hood picks up: a piece of litter slides in when the drag at the lips beats its friction on the road and
rises when the upward speed beats its terminal speed. The particle movie carries the litter classes through the
solved field (they ride the road into the hood at the sweeping speed).

*The fast run takes a minute or two on 4 cores (`processors=4` needs a working MPI — see docs/aeromant.md on the
conda-forge build); the fine one ~15 min. The notebook falls back to the recorded numbers
(`onager_sweeper_cfd.SUCTION`) if OpenFOAM is missing.*

In [ ]:
env = None
try:
    env = aeromant.OpenFOAMEnvironment.detect()
    print("OpenFOAM", env.version(), env.flavor())
except Exception as exc:
    print("no OpenFOAM:", exc)
CFD_QUALITY = "fast"                                   # "fine" for the 0.9 M-cell mesh (~15 min on 4 cores)
case = cfd.suction_case(cad["hood"].artifacts["stl"], RUNS / "cfd_hood", p, flow_rate=osr.FAN_FLOW, ground_speed=1.0, environment=env, quality=CFD_QUALITY)
prep = case.prepare(overwrite=True)
print(prep.ok, prep.messages, {k: prep.metrics[k] for k in ("body_bbox_min_m", "body_bbox_max_m", "background_cells")})
aviz.show(aviz.plot_setup(case))

In [ ]:
suction = dict(cfd.SUCTION)
cfd_res = None
if env is not None:
    cfd_res = case.run(processors=4, progress=True)
    print(cfd_res.ok, cfd_res.messages)
    if cfd_res.ok:
        suction = cfd.sample_hood(case, p)
        print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in suction.items()})
        print("mesh", cfd_res.metrics.get("mesh_cells"), "cells; converged", cfd_res.metrics.get("converged"), "in", cfd_res.metrics.get("iterations"), "iterations")
pd.DataFrame({"recorded (onager_sweeper_cfd.SUCTION)": cfd.SUCTION, "this run": suction}).drop("source")

In [ ]:
if cfd_res is not None and cfd_res.ok:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    aviz.plot_section(case, "U", normal="y", origin=(0, 0, 0), ax=axes[0], zoom=1.2)
    aviz.plot_section(case, "U", normal="x", origin=(p["hood_x"] / 1000, 0, 0), ax=axes[1], zoom=1.2)
    fig.tight_layout()
    aeromant.plot_residuals(case.workdir)

In [ ]:
pick = cfd.pickup_table(suction)
display(pick.round(2))
if cfd_res is not None and cfd_res.ok:
    movie_cfd, tracks = cfd.debris_movie(case, p, OUT / "onager_sweeper_suction.mp4", seconds=2.5, ground_speed=1.0)
    print("litter movie:", movie_cfd)
    display(tracks.summary())

## 5. FEA (Talos): the basket loaded, the hood under the depression; their modes

**Basket** (2 mm Al 5754, its own frame): the floor's underside held (it sits on the roof), 25 kg of rubbish on
the floor — the brick's drop from 0.4 m is a 2.3 kg impact on the sheet, taken as a dynamic factor 3 on its
weight, both on the floor face. **Hood** (6 mm Al, the CFD frame): the duct's top held in the hull, the fan's
depression on the roof (the outside pushes it in). Then the modes of both: the basket with its rubbish, the hood
with its duct — the broom turns at 2.5 Hz, the fan at 75 Hz.

In [ ]:
AL5754 = talos.Material("Al 5754-H22 sheet", youngs_modulus=70000.0, poissons_ratio=0.33, density=2.66e-9, yield_strength=130.0, source="EN 485-2")
L_b, W_b, H_b, t_b = p["basket_length"], p["basket_width"], p["basket_height"], p["basket_wall"]
BASKET_REGIONS = [talos.SurfacesOnPlane("underside", "z", 0.0), talos.SurfacesOnPlane("floor", "z", t_b)]
F_rubbish, F_brick = 25.0 * G, 3.0 * oss.OBJECTS["brick"]["mass"] * G
bm = talos.StructuralModel(cad["basket"].artifacts["step"], "mm-N-MPa", AL5754, BASKET_REGIONS, [talos.FixedSupport("underside")],
                           [talos.Force("floor", fz=-(F_rubbish + F_brick))], talos.MeshSettings(element_size=20.0, min_element_size=4.0), name="basket_load")
bm.mesh(RUNS / "basket_load", progress=True); basket_res = bm.solve(RUNS / "basket_load")
hx0, hy0, hz0, hx1, hy1, hz1 = design.hood_box(p)
duct_top = p["hull_bottom"] + 10.0
HOOD_REGIONS = [talos.SurfacesOnPlane("duct_top", "z", duct_top), talos.SurfacesOnPlane("roof", "z", hz1)]
dp = suction["fan_static_pressure_Pa"] * 1e-6                     # MPa: the depression under the roof
hm = talos.StructuralModel(cad["hood_shell"].artifacts["step"], "mm-N-MPa", AL5754, HOOD_REGIONS, [talos.FixedSupport("duct_top")],
                           [talos.Pressure("roof", dp)], talos.MeshSettings(element_size=12.0, min_element_size=3.0), name="hood_suction")
hm.mesh(RUNS / "hood_suction", progress=True); hood_res = hm.solve(RUNS / "hood_suction")
fea = pd.DataFrame({
    "basket: 25 kg of rubbish + the brick's drop (×3)": {"load": f"{F_rubbish + F_brick:.0f} N", "max von Mises [MPa]": basket_res.metrics.get("max_von_mises"), "SF yield": basket_res.metrics.get("safety_factor_yield"), "max displacement [mm]": basket_res.metrics.get("max_displacement")},
    f"hood: the fan's depression on the roof ({dp * 1e6:.0f} Pa)": {"load": f"{dp * (hx1 - hx0) * (hy1 - hy0):.1f} N", "max von Mises [MPa]": hood_res.metrics.get("max_von_mises"), "SF yield": hood_res.metrics.get("safety_factor_yield"), "max displacement [mm]": hood_res.metrics.get("max_displacement")},
}).T
fea.round(3)

In [ ]:
tviz.show(tviz.plot_results(basket_res, field="von_mises"))

In [ ]:
bmod = talos.StructuralModel(cad["basket"].artifacts["step"], "mm-N-MPa", AL5754, BASKET_REGIONS, [talos.FixedSupport("underside")], [],
                             talos.MeshSettings(element_size=25.0, min_element_size=5.0), name="basket_modal", masses=[talos.PointMass("floor", 25.0e-3)])
bmod.mesh(RUNS / "basket_modal"); basket_modes = bmod.solve_modes(RUNS / "basket_modal", n_modes=4)
hmod = talos.StructuralModel(cad["hood_shell"].artifacts["step"], "mm-N-MPa", AL5754, HOOD_REGIONS, [talos.FixedSupport("duct_top")], [],
                             talos.MeshSettings(element_size=15.0, min_element_size=4.0), name="hood_modal")
hmod.mesh(RUNS / "hood_modal"); hood_modes = hmod.solve_modes(RUNS / "hood_modal", n_modes=4)
f_basket, f_hood = basket_modes.metrics["frequencies_hz"], hood_modes.metrics["frequencies_hz"]
print(f"basket with 25 kg: {[round(f, 1) for f in f_basket]} Hz | hood on its duct: {[round(f, 1) for f in f_hood]} Hz")
tviz.show(tviz.plot_mode(basket_modes, mode=1))

## 6. Vibration: the broom's and the fan's unbalance

Two rotors run all day: the broom (9 kg disc, 150 rpm — a brush disc is balanced to a few mm) and the fan
(3 kg impeller, 4500 rpm, balance grade G 6.3). Their unbalance forces `m e ω²` excite the body at 2.5 Hz and
75 Hz; the structure answers with the transmissibility `1 / √((1 − r²)² + (2ζr)²)` of each mode (ζ 0.03, an input).
The chart puts the two forcing lines against the basket's and the hood's modes from §5 and the body on its tyres
(the Sentinel's quarter car: the ride frequency of the stiff stance on the tyres' 290 kN/m, notebook 20).

In [ ]:
m_disc, e_disc = osr.sweeper_masses(cad=cad_num)["broom disc (PA6, CAD)"] + osr.SWEEPER_KG["bristle ring (PP tufts)"], 2.0e-3   # inputs: disc mass, 2 mm eccentricity
m_imp, grade = 3.0, 6.3e-3                                                                  # fan impeller 3 kg, ISO 21940 G 6.3 (e·ω = 6.3 mm/s)
f_broom = osr.BROOM_RPM / 60; f_fan = 4500 / 60
F_broom = m_disc * e_disc * (2 * math.pi * f_broom) ** 2
e_fan = grade / (2 * math.pi * f_fan); F_fan = m_imp * e_fan * (2 * math.pi * f_fan) ** 2
k_tyre = 290e3; f_ride = math.sqrt(4 * k_tyre / M) / (2 * math.pi)
modes = {"body on its tyres (ride)": f_ride, "basket mode 1": f_basket[0], "basket mode 2": f_basket[1], "hood mode 1": f_hood[0], "hood mode 2": f_hood[1]}
zeta = 0.03
fs = np.logspace(-0.3, 2.5, 600)
fig, ax = plt.subplots(figsize=(10, 4.5))
for name, fn in modes.items():
    r = fs / fn
    ax.plot(fs, 1 / np.sqrt((1 - r ** 2) ** 2 + (2 * zeta * r) ** 2), label=f"{name}: {fn:.1f} Hz")
ax.axvline(f_broom, color="k", ls="--"); ax.text(f_broom, 20, f" broom {f_broom:.1f} Hz, {F_broom:.1f} N", fontsize=8)
ax.axvline(f_fan, color="r", ls="--"); ax.text(f_fan, 20, f" fan {f_fan:.0f} Hz, {F_fan:.1f} N", fontsize=8, color="r")
ax.set(xscale="log", yscale="log", xlabel="frequency [Hz]", ylabel="transmissibility", title="unbalance forcing vs the body's and the gear's modes (ζ 0.03)", ylim=(0.01, 30)); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
vib = pd.DataFrame({"forcing [Hz]": {"broom": f_broom, "fan": f_fan}, "unbalance force [N]": {"broom": F_broom, "fan": F_fan}})
for name, fn in modes.items():
    vib[f"T at {name} ({fn:.1f} Hz)"] = [1 / math.sqrt((1 - (f / fn) ** 2) ** 2 + (2 * zeta * f / fn) ** 2) for f in (f_broom, f_fan)]
vib.round(2)

## 7. The street in MuJoCo (ChironLab)

The Sweeper as a Chiron robot (`designs/onager_sweeper_robot.py`: the chassis's stiff braked legs, the Manus arm
servos, the broom as a velocity servo, the hood plates and the basket as collision geometry). The scene
(`designs/onager_sweeper_scenario.py`): asphalt, cans and packets in the hood's path (light props with hard contacts),
a brick and a box on the centre line. The **suction** is a scene hook: under the hood and around its lips it applies
the air drag of the field the CFD gave (§4's speeds, `onager_sweeper_cfd.SUCTION`) to each piece of litter; a piece
that reaches the duct is collected and logged. The mission (`designs/onager_sweeper_controller.py`): broom and fan
on, sweep at 1 m/s, stop at the brick, take it with the left pincer's hooked jaws (a hook grasp low on its sides),
swing it back over the side into the basket, release, stow; the box with the right pincer; sweep on. The objects'
poses are read from the simulation when an arm reaches (ideal perception); whether they stay in the jaws and land in
the basket is physics.

In [ ]:
scene = oss.Scene()
lab = oss.make_lab(scene, robot=robot, suction=suction, log_geoms=True)
ep = oss.run(lab, scene, duration=95.0)
ep.save(RUNS / "mission.npz")
ts = oss.timeseries(ep)
basket = {name: oss.in_basket(ep, name) for name, _, _ in scene.pickups}
print(f"mission finished: {ep.log['mission_finished']} | vacuumed {len(ep.log['collected'])} of {len(scene.litter)} pieces | in the basket: {basket} | robot at x = {ts.x.iloc[-1]:.2f} m")
display(oss.collected(ep).round(2).T)
phases = oss.phase_table(ep)
phases.round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
ax = axes[0, 0]
ax.plot(ts.t, ts.x, label="hull x [m]"); ax.plot(ts.t, ts.v * 10, label="speed × 10 [m/s]")
for name, t in ep.log["collected"].items():
    ax.axvline(t, color="#2e7d32", lw=0.6, alpha=0.6)
ax.set(xlabel="time [s]", title="the sweep (green: a piece of litter reaches the duct)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[0, 1]
for name, col in (("brick", "#6d4c41"), ("box", "#ef6c00")):
    ax.plot(ts.t, ts[f"{name}_z"], color=col, label=f"{name} height"); ax.plot(ts.t, ts[f"{name}_x"] - ts.x, color=col, ls=":", label=f"{name} x − hull x")
ax.set(xlabel="time [s]", ylabel="m", title="the pick-ups: lifted, carried back, dropped"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[1, 0]
ax.plot(ts.t, ts.broom_rpm, label="broom [rpm]"); ax.plot(ts.t, ts.tau_broom * 10, label="broom torque × 10 [N·m]"); ax.plot(ts.t, ts.tilt_deg * 100, label="hull tilt × 100 [deg]")
ax.set(xlabel="time [s]", title="broom and hull"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax = axes[1, 1]
for side, ls in (("L", "-"), ("R", "--")):
    for j, col in zip(("shoulder", "elbow", "wrist", "jaw_upper"), ("#1565c0", "#2e7d32", "#ef6c00", "#6d4c41")):
        key = "jaw" if j.startswith("jaw") else j
        ax.plot(ts.t, ts[f"tau_{side}_{j}"].abs() / act.get(omr.ARM_ACTUATORS[key]).stall_Nm, ls, color=col, lw=0.8, label=f"{side} {j}")
ax.axhline(1, color="k", lw=0.8); ax.set(xlabel="time [s]", ylabel="|τ| / stall", title="arm and jaw drives"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
for a in axes.flat:
    for t0, name, note in ep.log["mission"]:
        if note == "start": a.axvline(t0, color="#ddd", lw=0.5, zorder=0)
fig.tight_layout()

In [ ]:
dt = float(ep.log["t"][1] - ep.log["t"][0]); every = max(1, int(round(1 / (25 * dt))))
imgs = cviz.frames(ep, camera="follow", every=every, size=(960, 540))
movie = cviz.to_video(imgs, OUT / "onager_sweeper_street.mp4", fps=25)
print(f"movie: {movie} ({len(imgs)} frames)")
starts = {}
for t0, name, note in ep.log["mission"]:
    if note == "start": starts.setdefault(name, t0)
fig, axes = plt.subplots(2, 3, figsize=(18, 6.5))
for ax, (name, off) in zip(axes.flat, (("sweep to the brick", 3.0), ("L arm: jaws around the brick", 1.9), ("lift", 1.5),
                                       ("L arm: swing back to the basket", 3.9), ("release", 0.9), ("sweep on", 4.0))):
    j = min(len(imgs) - 1, int(round((starts.get(name, 0.0) + off) / (every * dt))))
    ax.imshow(imgs[j]); ax.set_axis_off(); ax.set_title(f"{name} (t = {j * every * dt:.1f} s)", fontsize=9)
fig.tight_layout()

## 8. Export

In [ ]:
doc = {"source": "23_onager_sweeper", "design": {"file": "designs/onager_sweeper.py", "parameters": p}, "job": JOB.to_dict(),
       "mass_kg": M, "mass_budget_kg": budget, "sweeping_gear_kg": gear.iloc[:, 0].to_dict(), "power_W": power["W"].to_dict(), "endurance_h": hours,
       "arms": rows, "suction": suction, "pickup": pick.to_dict(orient="index"), "cfd_metrics": (cfd_res.metrics if cfd_res is not None else None),
       "fea": fea.to_dict(orient="index"), "basket_modes_hz": f_basket, "hood_modes_hz": f_hood, "vibration": vib.to_dict(orient="index"),
       "mission": {"finished": ep.log["mission_finished"], "events": ep.log["events"], "vacuumed": ep.log["collected"], "in_basket": basket,
                   "phases": phases.to_dict(orient="index"), "movie": str(movie)}}
(RUNS / "onager_sweeper.json").write_text(json.dumps(doc, indent=2, default=float))
print("written:", RUNS / "onager_sweeper.json")

**What the numbers say** — read them above; the ones that set the design: the mass budget (the series' 3 mm skin
on this body is a quarter of the machine), the fan's pressure and the endurance it costs (the downstream losses,
not the hood, decide the fan), the pick-up table of §4 (the hood lifts leaves, paper, butts and caps; gravel and a
crushed can need the duct's speed — they go up only once the broom has brought them under the duct; a 15 mm
pebble stays for the broom), the basket under its load, the two unbalance lines well away from the modes, and in
§7 which litter the hood took and whether the brick and the box landed in the basket.

**Next steps an engineer would take:** the broom's own flow (a spinning disc throws air and litter outward: a
rotating-wall boundary in the CFD, or the MRF template); the hopper, cyclone and filter losses measured instead of
assumed; a brush-pressure control (the bristle load sets the broom's power and wear); perception for the pick-ups
instead of the simulated poses; a wrist roll so a brick lying askew is taken square; the basket's own lid and a
tip-out mechanism; wet sweeping (a water spray for the dust) and the mass of its tank.